# TFT (Temporal Fusion Transformer) Training
**Train on Kaggle GPU → download weights for local CPU inference**

## Architecture
- Variable Selection Networks (VSN) — learns which indicators matter per timestep
- BiLSTM encoder — captures sequential dependencies
- Interpretable Multi-Head Attention — temporal self-attention
- Gated residual connections throughout

## Instructions
1. Enable GPU: Settings → Accelerator → **GPU T4 x2** or P100
2. Run All (~35-45 min)
3. Download from Output panel: `tft_weights.pth`, `tft_config.json`, `feature_scaler.pkl`
4. Place all 3 in your local `models/pretrained/`
5. Restart the Streamlit app — TFT auto-activates

In [ ]:
!pip install yfinance ta joblib scikit-learn -q

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import math
import json
import joblib
import warnings
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import RobustScaler
from sklearn.utils.class_weight import compute_class_weight
import yfinance as yf
import ta
warnings.filterwarnings('ignore')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

# ─── CONFIG (MUST MATCH models/tft_model.py) ───────────────────────────────
SEQ_LEN       = 60
D_MODEL       = 64
N_HEADS       = 4
N_LSTM_LAYERS = 2
DROPOUT       = 0.2
NUM_CLASSES   = 3
BATCH_SIZE    = 256
NUM_EPOCHS    = 120
LR            = 0.001
BUY_THRESH    = 0.05
SELL_THRESH   = -0.05
HORIZON       = 30

TICKERS = [
    'RELIANCE.NS','TCS.NS','INFY.NS','HDFCBANK.NS','ICICIBANK.NS',
    'HINDUNILVR.NS','SBIN.NS','BAJFINANCE.NS','BHARTIARTL.NS','WIPRO.NS',
    'AXISBANK.NS','LT.NS','MARUTI.NS','SUNPHARMA.NS','M&M.NS',
    'KOTAKBANK.NS','ITC.NS','HCLTECH.NS','ASIANPAINT.NS','TITAN.NS',
    'ULTRACEMCO.NS','BAJAJFINSV.NS','POWERGRID.NS','NTPC.NS','ONGC.NS',
    'DRREDDY.NS','DIVISLAB.NS','CIPLA.NS','TECHM.NS','NESTLEIND.NS',
    'JSWSTEEL.NS','TATASTEEL.NS','HINDALCO.NS','COALINDIA.NS','GRASIM.NS',
    'PIIND.NS','PERSISTENT.NS','COFORGE.NS','MPHASIS.NS','KPITTECH.NS',
    'CHOLAFIN.NS','FEDERALBNK.NS','IDFCFIRSTB.NS','RBLBANK.NS','BANDHANBNK.NS',
    'ZYDUSLIFE.NS','TORNTPHARM.NS','ALKEM.NS','ABBOTINDIA.NS','LICHSGFIN.NS',
]
print(f'Training on {len(TICKERS)} NSE stocks')

In [ ]:
# ─── DATA PIPELINE ─────────────────────────────────────────────────────────
def build_target(close, horizon):
    ret = close.shift(-horizon) / close - 1.0
    y = pd.Series(1, index=close.index)
    y[ret > BUY_THRESH] = 2
    y[ret < SELL_THRESH] = 0
    y[ret.isna()] = np.nan
    return y

def fetch(ticker):
    try:
        df = yf.download(ticker, period='10y', interval='1d', auto_adjust=True, progress=False)
        if df is None or df.empty or len(df) < 500:
            return None, None, None
        
        # Flatten MultiIndex if present in newer yfinance versions
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = [col[0] if isinstance(col, tuple) else col for col in df.columns]
        
        # Extract 1D Series for each price component
        c = pd.Series(df['Close'].values.flatten(), index=df.index, name='Close', dtype=float)
        h = pd.Series(df['High'].values.flatten(), index=df.index, name='High', dtype=float)
        lo = pd.Series(df['Low'].values.flatten(), index=df.index, name='Low', dtype=float)
        v = pd.Series(df['Volume'].values.flatten(), index=df.index, name='Volume', dtype=float)
        
        feat = pd.DataFrame(index=df.index)
        for w in [1, 5, 10, 20, 60]:
            feat[f'ret_{w}d'] = np.log(c / c.shift(w))
        for w in [10, 20, 60]:
            feat[f'vol_{w}d'] = c.pct_change().rolling(w).std()
            
        feat['SMA_20'] = ta.trend.sma_indicator(c, 20)
        feat['SMA_50'] = ta.trend.sma_indicator(c, 50)
        feat['SMA_200']= ta.trend.sma_indicator(c, 200)
        feat['EMA_12'] = ta.trend.ema_indicator(c, 12)
        feat['EMA_26'] = ta.trend.ema_indicator(c, 26)
        feat['price_vs_sma50']  = (c - feat['SMA_50'])  / feat['SMA_50'].replace(0, np.nan)
        feat['price_vs_sma200'] = (c - feat['SMA_200']) / feat['SMA_200'].replace(0, np.nan)
        feat['RSI_14']     = ta.momentum.rsi(c, 14)
        feat['StochRSI']   = ta.momentum.stochrsi(c, 14)
        feat['Williams_R'] = ta.momentum.williams_r(h, lo, c)
        
        macd = ta.trend.MACD(c)
        feat['MACD'] = macd.macd()
        feat['MACD_Sig'] = macd.macd_signal()
        feat['MACD_Hist'] = macd.macd_diff()
        
        bb = ta.volatility.BollingerBands(c)
        feat['BB_PctB'] = bb.bollinger_pband()
        feat['ATR_14']  = ta.volatility.average_true_range(h, lo, c, 14)
        feat['ADX_14']  = ta.trend.adx(h, lo, c, 14)
        feat['CCI_20']  = ta.trend.cci(h, lo, c, 20)
        feat['Vol_Ratio']  = v / v.rolling(20).mean()
        feat['OBV_roc']    = ta.volume.on_balance_volume(c, v).pct_change(20)
        feat['drawdown_52w'] = (c - c.rolling(252).max()) / c.rolling(252).max().replace(0, np.nan)
        
        feat.replace([np.inf, -np.inf], np.nan, inplace=True)
        feat.ffill(inplace=True)
        feat.fillna(0, inplace=True)
        
        y = build_target(c, HORIZON)
        fcols = list(feat.columns)
        return feat, y, fcols
    except Exception as e:
        print(f'  Failed {ticker}: {e}')
        return None, None, None

all_X, all_y, feature_cols = [], [], None
for t in TICKERS:
    print(f'Fetching {t}...')
    X, y, fcols = fetch(t)
    if X is None:
        continue
    if feature_cols is None:
        feature_cols = fcols
    all_X.append(X)
    all_y.append(y)

print(f'\nLoaded {len(all_X)} stocks | features: {len(feature_cols) if feature_cols else 0}')

In [ ]:
# ─── BUILD SEQUENCES ───────────────────────────────────────────────────────
def build_seqs(X, y, seq_len=SEQ_LEN):
    seqs, labels = [], []
    Xv, yv = X.values.astype(np.float32), y.values
    for i in range(seq_len, len(Xv)):
        if np.isnan(yv[i]):
            continue
        seqs.append(Xv[i-seq_len:i])
        labels.append(int(yv[i]))
    return np.array(seqs), np.array(labels)

Xs, ys = [], []
for X, y in zip(all_X, all_y):
    s, l = build_seqs(X, y)
    Xs.append(s)
    ys.append(l)

X_all = np.concatenate(Xs)
y_all = np.concatenate(ys)
print(f'Total sequences: {len(X_all):,}  Shape: {X_all.shape}')
print(f'Classes — Sell:{np.sum(y_all==0):,}  Hold:{np.sum(y_all==1):,}  Buy:{np.sum(y_all==2):,}')

# Scale
n, s, f = X_all.shape
scaler = RobustScaler()
X_scaled = scaler.fit_transform(X_all.reshape(-1, f)).reshape(n, s, f)
X_scaled = np.nan_to_num(X_scaled, nan=0.0, posinf=0.0, neginf=0.0)
joblib.dump(scaler, '/kaggle/working/feature_scaler.pkl')
print('Scaler saved.')

split = int(0.85 * n)
X_train, X_val = X_scaled[:split], X_scaled[split:]
y_train, y_val = y_all[:split], y_all[split:]
print(f'Train: {len(X_train):,}  Val: {len(X_val):,}')

In [ ]:
# ─── TFT ARCHITECTURE (MUST MATCH models/tft_model.py) ────────────────────
class GatedLinearUnit(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.fc = nn.Linear(d, d*2)
        self.gate = nn.Sigmoid()
    def forward(self, x):
        o, g = self.fc(x).chunk(2, dim=-1)
        return o * self.gate(g)

class AddNorm(nn.Module):
    def __init__(self, d, dr=0.1):
        super().__init__()
        self.glu = GatedLinearUnit(d)
        self.norm = nn.LayerNorm(d)
        self.drop = nn.Dropout(dr)
    def forward(self, x, res):
        return self.norm(self.drop(self.glu(x)) + res)

class GatedResidualNetwork(nn.Module):
    def __init__(self, d, dh=None, dr=0.1):
        super().__init__()
        dh = dh or d
        self.fc1 = nn.Linear(d, dh)
        self.fc2 = nn.Linear(dh, d)
        self.elu = nn.ELU()
        self.an = AddNorm(d, dr)
    def forward(self, x):
        return self.an(self.fc2(self.elu(self.fc1(x))), x)

class VariableSelectionNetworkV2(nn.Module):
    def __init__(self, n, d, dr=0.1):
        super().__init__()
        self.wn = nn.Sequential(nn.Linear(n, d), nn.ELU(), nn.Dropout(dr), nn.Linear(d, n))
        self.sm = nn.Softmax(dim=-1)
    def forward(self, x):
        w = self.sm(self.wn(x))
        return x * w, w.mean(dim=1)

class InterpretableMultiHeadAttention(nn.Module):
    def __init__(self, d, nh=4, dr=0.1):
        super().__init__()
        self.nh = nh
        self.dk = d // nh
        self.q = nn.Linear(d, d)
        self.k = nn.Linear(d, d)
        self.v = nn.Linear(d, d)
        self.out = nn.Linear(d, d)
        self.drop = nn.Dropout(dr)
    def forward(self, x):
        B, T, D = x.shape
        H, dk = self.nh, self.dk
        Q = self.q(x).view(B, T, H, dk).transpose(1, 2)
        K = self.k(x).view(B, T, H, dk).transpose(1, 2)
        V = self.v(x).view(B, T, H, dk).transpose(1, 2)
        attn = torch.softmax(torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(dk), dim=-1)
        attn = self.drop(attn)
        out = torch.matmul(attn, V).transpose(1, 2).contiguous().view(B, T, D)
        return self.out(out), attn.mean(dim=1)

class TemporalFusionTransformer(nn.Module):
    def __init__(self, nf, d=D_MODEL, nh=N_HEADS, nl=N_LSTM_LAYERS, dr=DROPOUT, nc=NUM_CLASSES):
        super().__init__()
        self.ip = nn.Linear(nf, d)
        self.vsn = VariableSelectionNetworkV2(d, d, dr)
        self.bilstm = nn.LSTM(d, d//2, nl, batch_first=True, bidirectional=True, dropout=dr if nl > 1 else 0)
        self.an1 = AddNorm(d, dr)
        self.attn = InterpretableMultiHeadAttention(d, nh, dr)
        self.an2 = AddNorm(d, dr)
        self.ff = GatedResidualNetwork(d, d*2, dr)
        self.an3 = AddNorm(d, dr)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.drop = nn.Dropout(dr)
        self.cls = nn.Sequential(nn.Linear(d, d//2), nn.GELU(), nn.Dropout(dr), nn.Linear(d//2, nc))
    def forward(self, x):
        B, T, F = x.shape
        x = self.ip(x)
        x, _ = self.vsn(x)
        lo, _ = self.bilstm(x)
        x = self.an1(lo, x)
        ao, _ = self.attn(x)
        x = self.an2(ao, x)
        fo = self.ff(x.reshape(B*T, -1)).reshape(B, T, -1)
        x = self.an3(fo, x)
        p = self.pool(x.transpose(1, 2)).squeeze(-1)
        return self.cls(self.drop(p))

INPUT_SIZE = f
model = TemporalFusionTransformer(nf=INPUT_SIZE).to(DEVICE)
print(f'TFT parameters: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
# ─── TRAINING ──────────────────────────────────────────────────────────────
cw = compute_class_weight('balanced', classes=np.array([0, 1, 2]), y=y_train)
print(f'Class weights: Sell={cw[0]:.2f} Hold={cw[1]:.2f} Buy={cw[2]:.2f}')
criterion = nn.CrossEntropyLoss(weight=torch.FloatTensor(cw).to(DEVICE))
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-5)

train_ds = TensorDataset(torch.FloatTensor(X_train), torch.LongTensor(y_train))
val_ds   = TensorDataset(torch.FloatTensor(X_val),   torch.LongTensor(y_val))
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  pin_memory=True, num_workers=2)
val_dl   = DataLoader(val_ds,   batch_size=BATCH_SIZE*2, shuffle=False, pin_memory=True, num_workers=2)

def evaluate(dl):
    model.eval()
    correct = total = loss_sum = 0
    with torch.no_grad():
        for xb, yb in dl:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            out = model(xb)
            loss_sum += criterion(out, yb).item() * len(yb)
            correct += (out.argmax(1) == yb).sum().item()
            total += len(yb)
    return correct / total, loss_sum / total

best_val, patience_count, PATIENCE = 0.0, 0, 15
print('\nTraining TFT...')
for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    for xb, yb in train_dl:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
    scheduler.step()
    val_acc, val_loss = evaluate(val_dl)
    if epoch % 10 == 0 or epoch == 1:
        print(f'Epoch {epoch:03d} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}')
    if val_acc > best_val:
        best_val = val_acc
        torch.save(model.state_dict(), '/kaggle/working/tft_weights.pth')
        patience_count = 0
    else:
        patience_count += 1
        if patience_count >= PATIENCE:
            print(f'Early stop @ epoch {epoch}')
            break
print(f'Best Val Acc: {best_val:.4f}')

In [ ]:
# ─── EVALUATE DIRECTION ACCURACY ───────────────────────────────────────────
model.load_state_dict(torch.load('/kaggle/working/tft_weights.pth', map_location=DEVICE))
model.eval()
all_p, all_l = [], []
with torch.no_grad():
    for xb, yb in val_dl:
        all_p.extend(model(xb.to(DEVICE)).argmax(1).cpu().numpy())
        all_l.extend(yb.numpy())
all_p, all_l = np.array(all_p), np.array(all_l)
ov_acc = (all_p == all_l).mean()
dm = (all_p != 1) & (all_l != 1)
dir_acc = (all_p[dm] == all_l[dm]).mean() if dm.sum() else 0
print(f'Overall Accuracy:    {ov_acc:.4f} ({ov_acc*100:.1f}%)')
print(f'Directional Accuracy:{dir_acc:.4f} ({dir_acc*100:.1f}%)')
for cls, name in [(0, 'Sell'), (1, 'Hold'), (2, 'Buy')]:
    m = all_l == cls
    if m.sum():
        print(f'  {name} Recall: {(all_p[m] == cls).mean():.4f}')

In [ ]:
# ─── SAVE CONFIG ───────────────────────────────────────────────────────────
config = {
    'n_features': INPUT_SIZE,
    'd_model': D_MODEL,
    'n_heads': N_HEADS,
    'n_lstm_layers': N_LSTM_LAYERS,
    'dropout': DROPOUT,
    'seq_len': SEQ_LEN,
    'num_classes': NUM_CLASSES,
    'val_accuracy': round(float(best_val), 4),
    'dir_accuracy': round(float(dir_acc), 4),
    'feature_cols': feature_cols,
    'horizon_days': HORIZON,
    'train_tickers': TICKERS,
}
with open('/kaggle/working/tft_config.json', 'w') as f:
    json.dump(config, f, indent=2)
print('\nSaved:')
print('  /kaggle/working/tft_weights.pth')
print('  /kaggle/working/tft_config.json')
print('  /kaggle/working/feature_scaler.pkl')
print('\nCopy all 3 to: models/pretrained/')
print('Restart Streamlit — TFT auto-activates!')